In [ ]:
#| default_exp handlers.pipeline.loader

# Loader

`HandlerConfig`, `load_data`, and `gap_check` — the first gate in the GeneralHandler pipeline.

In [ ]:
#| export
from __future__ import annotations
import io
from pathlib import Path
from typing import Optional
import requests
import yaml
import pandas as pd
from pydantic import BaseModel, Field

## HandlerConfig

Pydantic model that binds every YAML field with types; `from_yaml` is the single entry point.

In [ ]:
#| export
class PluginSpec(BaseModel):
    "External Callback injection spec: fully-qualified class path + constructor kwargs."
    path: str                           # e.g. 'marisco.handlers.tepco.RemoveJapaneseCharCB'
    args: dict = Field(default_factory=dict)

class MeltEntry(BaseModel):
    "One wide-to-long mapping entry: value column, uncertainty column, nuclide, unit, and lab."
    val:     str
    unc:     str
    nuclide: str
    unit:    str
    lab:     str

class UnitConversionCfg(BaseModel):
    "Single unit-conversion rule with its physical factor and optional metadata."
    nuclide:     str
    src_unit:    str
    dst_unit:    str
    factor:      float
    factor_name: str = ""
    comment:     str = ""

class HandlerConfig(BaseModel):
    "Complete handler configuration loaded from a YAML data-contract file."
    # handler
    module_name: str
    title:       str = ""
    description: str = ""
    # data_source
    url:       str
    fname_out: str
    zenodo_id: str = ""
    fmt:       str = "csv"
    # rename_cols (legacy) + declarative columns shorthand (S-7c)
    rename:      dict[str, str] = Field(default_factory=dict)
    string_cast: list[str]      = Field(default_factory=list)
    columns:     dict[str, str] = Field(default_factory=dict)  # provider_col → MARIS_col, merged into rename at pipeline build time
    # case normalisation: {src_col: dst_col} → LowerStripNameCB auto-assembled before pre_cbs (S-7c)
    normalize_case: dict[str, str] = Field(default_factory=dict)
    # parse_datetime
    col_date:    Optional[str] = None
    col_time:    Optional[str] = None
    dt_format:   str           = "%Y-%m-%d"
    time_format: Optional[str] = None  # explicit format hint; overrides dt_format when set
    # melt
    meta_cols: list[str]             = Field(default_factory=list)
    melt_spec: list[MeltEntry]       = Field(default_factory=list)
    # unit_conversions
    unit_conversions: list[UnitConversionCfg] = Field(default_factory=list)
    # nomenclatures (all optional — empty dict/0 → Null-Object no-ops in build_core_pipeline)
    nuclide_lut:   dict[str, int] = Field(default_factory=dict)
    unit_lut:      dict[str, int] = Field(default_factory=dict)
    lab_lut:       dict[str, int] = Field(default_factory=dict)
    lab_constants: dict[str, str] = Field(default_factory=dict)
    area_default:  int            = 0
    # output
    keywords: list[str] = Field(default_factory=list)
    # plugin injection: custom loader + pre/post standard core pipeline
    loader:   Optional[PluginSpec] = None
    pre_cbs:  list[PluginSpec]     = Field(default_factory=list)
    post_cbs: list[PluginSpec]     = Field(default_factory=list)

    @classmethod
    def from_yaml(cls, path: str | Path) -> "HandlerConfig":
        "Load and validate a handler YAML config; raises ValidationError on schema mismatch."
        raw      = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
        h, ds    = raw["handler"], raw["data_source"]
        nom      = raw.get("nomenclatures", {})   # optional: absent → all LUTs default to {}
        ren      = raw.get("rename_cols", {})
        pdt      = raw.get("parse_datetime", {})
        mlt      = raw.get("melt", {})
        loader_raw = raw.get("loader", None)
        return cls(
            module_name      = h["module_name"],
            title            = h.get("title", ""),
            description      = h.get("description", "").strip(),
            url              = ds["url"],
            fname_out        = ds["fname_out"],
            zenodo_id        = ds.get("zenodo_id", ""),
            fmt              = ds.get("format", "csv"),
            rename           = ren.get("mapping", {}),
            string_cast      = ren.get("string_cast", []),
            columns          = raw.get("columns", {}),
            normalize_case   = raw.get("normalize_case", {}),
            col_date         = pdt.get("col_date"),
            col_time         = pdt.get("col_time"),
            dt_format        = pdt.get("format", "%Y-%m-%d"),
            time_format      = raw.get("time_format"),
            meta_cols        = mlt.get("meta_cols", []),
            melt_spec        = mlt.get("spec", []),
            unit_conversions = raw.get("unit_conversions", []),
            nuclide_lut      = nom.get("nuclide_lut", {}),
            unit_lut         = nom.get("unit_lut", {}),
            lab_lut          = nom.get("lab_lut", {}),
            lab_constants    = nom.get("lab_constants", {}),
            area_default     = nom.get("area_default", 0),
            keywords         = raw.get("output", {}).get("keywords", []),
            loader           = PluginSpec(**loader_raw) if loader_raw else None,
            pre_cbs          = raw.get("pre_cbs", []),
            post_cbs         = raw.get("post_cbs", []),
        )

In [ ]:
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
print(f"module_name:       {cfg.module_name}")
print(f"url:               {cfg.url[:55]}...")
print(f"rename keys:       {list(cfg.rename)[:3]}")
print(f"melt_spec entries: {len(cfg.melt_spec)}  (first: {cfg.melt_spec[0].val!r})")
print(f"unit_conversions:  {len(cfg.unit_conversions)}  (factor: {cfg.unit_conversions[0].factor:.3e})")
print(f"nuclide_lut:       {cfg.nuclide_lut}")
print(f"col_date / fmt:    {cfg.col_date!r} / {cfg.dt_format!r}")
print("HandlerConfig.from_yaml ✓")

In [ ]:
# PluginSpec: pre_cbs/post_cbs default to empty lists
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
assert cfg.pre_cbs  == [], f"Expected [], got {cfg.pre_cbs}"
assert cfg.post_cbs == [], f"Expected [], got {cfg.post_cbs}"

# PluginSpec from dict (Pydantic coercion)
spec = PluginSpec(path="marisco.callbacks.SoftShiftLonCB", args={"shift": 180.0})
assert spec.path == "marisco.callbacks.SoftShiftLonCB"
assert spec.args == {"shift": 180.0}
print("PluginSpec ✓ — pre_cbs/post_cbs default [], dict coercion ✓")

## load_data

Lazy intake — fetches the provider CSV/TSV and wraps it in the `{grp: DataFrame}` contract.

In [ ]:
#| export
def load_data(cfg: HandlerConfig, grp: str = "SEAWATER") -> dict[str, pd.DataFrame]:
    "Fetch raw CSV/TSV from cfg.url and return {grp: DataFrame}."
    r = requests.get(cfg.url, timeout=60)
    r.raise_for_status()
    sep = "	" if cfg.fmt == "tsv" else ","
    return {grp: pd.read_csv(io.BytesIO(r.content), sep=sep)}

## gap_check

Fail-Fast sensor: raises `ValueError` with scaffold CBs when required MARIS columns will be absent.

In [ ]:
#| export
_MARIS_REQUIRED = frozenset({"LAT", "LON", "TIME", "NUCLIDE", "VALUE", "UNC", "UNIT"})
_MELT_PROVIDES  = frozenset({"NUCLIDE", "VALUE", "UNC", "UNIT"})

def gap_check(cfg: HandlerConfig) -> None:
    "Fail-Fast: raise ValueError listing missing MARIS columns and printing skeleton CBs."
    if cfg.pre_cbs: return  # plugin chain is responsible for column provision
    # columns (S-7c shorthand) and rename both contribute mapped destination names
    all_rename = {**cfg.columns, **cfg.rename}
    available = (set(all_rename.values())
                 | (_MELT_PROVIDES if cfg.melt_spec else set())
                 | ({"TIME"}       if cfg.col_date  else set()))
    gaps = _MARIS_REQUIRED - available
    if not gaps:
        return
    skeleton = "\n\n".join(
        f"class Fill{g}CB(PerGroupCB):\n"
        f"    \"TODO: provide {g} — add to columns/rename_cols or as a standalone CB.\"\n"
        f"    grps = [\'SEAWATER\']\n"
        f"    def each_grp(self, grp, df, tfm): df[\'{g}\'] = None  # FIXME"
        for g in sorted(gaps)
    )
    print(f"\n⚠  GAP in {cfg.title!r} — missing MARIS columns: {sorted(gaps)}\n\n{skeleton}\n")
    raise ValueError(f"YAML spec missing mappings for: {sorted(gaps)}")

In [ ]:
# Valid config: all 7 MARIS columns covered by rename + melt + parse_datetime
cfg_ok = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg_ok)
print("gap_check(fram_strait) → passed ✓")

In [ ]:
# Minimal config with no columns at all: fires with skeleton CBs + ValueError
cfg_bad = HandlerConfig(
    module_name="test.handler",
    title="Minimal Test",
    url="http://example.com/data.csv",
    fname_out="test.nc",
)
try:
    gap_check(cfg_bad)
except ValueError as e:
    print(f"
ValueError raised ✓: {e}")